# Parlianote — NLP Analysis Notebook

**Team Phoenix (Group 11)**

Postcode -> MP -> topics -> grounded summaries, built on the
[Australian Parliamentary Debates Database (1998-2025)](https://zenodo.org/records/17351233)
(Katz & Alexander, University of Toronto) — Hansard transcripts from the
Australian House of Representatives.

This notebook is organised by technique. Each section has: **what** we did,
**code**, and a **markdown insights cell** summarising what we actually found —
read those insight cells first if you just want the takeaways for the poster.

**Techniques covered:**
1. Data cleaning / preprocessing
2. Postcode -> electorate -> MP lookup
3. LDA topic modelling (unsupervised)
4. Zero-shot issue classification (embeddings + calibration)
5. RAG grounded summarisation (FAISS + sentence embeddings)
6. Evaluation (manual accuracy check, retrieval precision, faithfulness check)


## 0. Setup

Run this cell once per environment. If a package is missing it will install it.

In [ ]:
# --- Install (safe to re-run; skips if already installed) ---
import importlib
for pkg in ["gensim", "sentence_transformers", "faiss", "nltk", "pyarrow"]:
    if importlib.util.find_spec(pkg) is None:
        %pip install -q gensim sentence-transformers faiss-cpu pyarrow nltk

# --- Core ---
import os
import re
import pandas as pd
import numpy as np

# --- Topic modelling (LDA) ---
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import nltk
from nltk.corpus import stopwords

# --- Embeddings, semantic search, RAG ---
from sentence_transformers import SentenceTransformer
import faiss

nltk.download("stopwords", quiet=True)

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)

print("Setup complete.")

## 1. Load and clean the Hansard corpus

**Technique: data cleaning / preprocessing.**

Raw dataset: [Australian Parliamentary Debates Database, 1998-2025](https://zenodo.org/records/17351233).
It contains every sitting-day proceeding of the House of Representatives — including
procedural text (e.g. "Mr SPEAKER took the chair...") and interjections, which are not
real MP speeches.

We keep only rows that are: tagged with an electorate, flagged as an actual member
speaking, **not** an interjection, and have real body text (> 30 characters).

**Path note:** this notebook expects the folder layout
`NLP AT2/notebooks/parlianote_analysis.ipynb` and `NLP AT2/data/corpus_1998_to_2025.parquet`
(i.e. `data/` sits next to `notebooks/`). Adjust `RAW_PATH` below if your layout differs.

In [ ]:
RAW_PATH = "../data/corpus_1998_to_2025.parquet"

assert os.path.exists(RAW_PATH), (
    f"Could not find {RAW_PATH}. Make sure corpus_1998_to_2025.parquet is in the "
    "data/ folder next to notebooks/, or update RAW_PATH above."
)

df = pd.read_parquet(RAW_PATH)
print(f"Raw rows: {len(df):,}")

# keep only real member speeches: has an electorate, is flagged as a member,
# not an interjection, and has actual body text
clean = df[
    df["electorate"].notna()
    & (df["member"] == "1")
    & (df["interject"] != "1")
    & df["body"].notna()
    & (df["body"].str.strip().str.len() > 30)
].copy()

clean["date"] = pd.to_datetime(clean["date"])
clean["year"] = clean["date"].dt.year
clean["word_count"] = clean["body"].str.split().str.len()

keep_cols = ["date", "year", "displayName", "uniqueID", "electorate",
             "partyAbbrev", "partyName", "gender", "question", "answer", "body", "word_count"]
clean = clean[keep_cols]

print(f"Clean speeches: {len(clean):,}")
print(f"Date range: {clean['date'].min().date()} to {clean['date'].max().date()}")
print(f"Unique electorates: {clean['electorate'].nunique()}")
print(f"Unique speakers: {clean['uniqueID'].nunique()}")
clean.head()

In [ ]:
# quick exploratory stats - useful context for the poster
print("=== Top 10 electorates by speech count ===")
print(clean["electorate"].value_counts().head(10))

print("\n=== Party breakdown ===")
print(clean["partyAbbrev"].value_counts().head(8))

print("\n=== Speech word-count distribution ===")
print(clean["word_count"].describe())

### Insights — Section 1

- Filtering 647,852 raw rows down to real MP speeches gives **362,203 clean speeches**
  spanning **1998-03-02 to 2025-07-31** (27+ years), across **166 electorates** and
  **421 individual speakers**.
- The two major parties dominate speech volume almost 1:1 (**ALP ~161,657** vs
  **LIB ~154,323**), dwarfing minor parties (Greens ~1,771) — reflecting two-party
  dominance of the chamber even though minor-party/independent voices can be
  disproportionately vocal per member.
- The most active electorates by raw speech count (Grayndler, Warringah, Bennelong...)
  tend to be held by senior figures or high-profile independents — e.g. Warringah's
  volume is driven by independent MP Zali Steggall, who we use later as our worked
  example for the grounded-summary demo.
- Speech length is highly skewed (median ~204 words, mean ~465 words) — most speeches
  are short procedural contributions, with a long tail of substantial ones. We filter
  to 80-3000 words later for topic work, to avoid one-line interjections and outlier
  mega-speeches skewing the model.

### Data correction made after this notebook (see the technical report, Section 3.4)

The cleaning filter above keeps rows with `member == "1"`. The source data leaves `member` blank for MPs elected from 2022 (and also leaves their `uniqueID` blank), so this filter silently drops 8,658 genuine speeches from 69 MPs. The web application and all later analysis use `member != "0"` (370,861 speeches) and the display name as a fallback MP key. The numbers in this notebook (362,203 speeches; LDA coherence 0.459; issue-classifier results) were produced with the original filter and are kept as the phase-1 record; `src/train_lda.py` reproduces the LDA result exactly.

## 2. Postcode -> Electorate -> current MP

**Technique: data integration (not NLP itself, but required for the "type your postcode"
feature).** The Hansard dataset only records *electorate names* (e.g. "Warringah"), not
postcodes — so we need a second, independent dataset to bridge postcode -> electorate.

We use the free, open [Australian Postcodes dataset](https://www.matthewproctor.com/australian_postcodes)
(no login required), which includes a `CED` / `electorate` column sourced from the AEC.

Download it once and save as `data/postcode_electorate.csv` (same folder as the parquet file).

In [ ]:
POSTCODE_PATH = "../data/postcode_electorate.csv"

if not os.path.exists(POSTCODE_PATH):
    print("Downloading postcode -> electorate lookup...")
    import urllib.request
    # the host blocks Python's default urllib user-agent (403), so pretend to be a browser
    req = urllib.request.Request(
        "https://www.matthewproctor.com/Content/postcodes/australian_postcodes.csv",
        headers={"User-Agent": "Mozilla/5.0"},
    )
    with urllib.request.urlopen(req) as resp, open(POSTCODE_PATH, "wb") as out_file:
        out_file.write(resp.read())
    print("Downloaded.")

pc = pd.read_csv(POSTCODE_PATH, dtype=str)
pc = pc[["postcode", "locality", "state", "electorate"]].dropna(subset=["electorate"])
postcode_lookup = pc.drop_duplicates(subset=["postcode", "electorate"])

print(f"Postcode -> electorate rows: {len(postcode_lookup):,}")
print(f"Unique postcodes: {postcode_lookup['postcode'].nunique():,}")

# how well does this cover our Hansard electorates?
hansard_electorates = set(clean["electorate"].dropna().unique())
csv_electorates = set(postcode_lookup["electorate"].dropna().unique())
overlap = hansard_electorates & csv_electorates
print(f"\nOverlap with Hansard electorates: {len(overlap)} / {len(hansard_electorates)} "
      f"({len(overlap)/len(hansard_electorates):.0%})")
print("In Hansard but not current postcode data (renamed/abolished seats):",
      sorted(hansard_electorates - csv_electorates)[:10])

In [ ]:
# current MP per electorate = speaker with the most recent speech in that electorate
speeches_sorted = clean.sort_values("date")
current_mp = speeches_sorted.groupby("electorate").tail(1)[
    ["electorate", "displayName", "partyAbbrev", "date"]
].rename(columns={"date": "last_seen"})

def postcode_to_mp(postcode: str):
    """Look up the current MP for a given postcode."""
    rows = postcode_lookup[postcode_lookup["postcode"] == postcode]
    if rows.empty:
        return f"No electorate match for postcode {postcode}"
    results = []
    for electorate in rows["electorate"].unique():
        mp_row = current_mp[current_mp["electorate"] == electorate]
        if mp_row.empty:
            results.append(f"{electorate}: no MP record in Hansard data")
        else:
            r = mp_row.iloc[0]
            results.append(f"{electorate}: {r['displayName']} ({r['partyAbbrev']}), last seen {r['last_seen'].date()}")
    return results

# demo with real postcodes
for p in ["2088", "2060", "3000", "4000", "6000", "2600"]:
    print(f"{p} ->", postcode_to_mp(p))

### Insights — Section 2

- The postcode lookup covers **143 of 166** Hansard electorates (**86%**) — the gap is
  entirely explained by electorate redistributions over 27 years (seats renamed or
  abolished, e.g. Batman -> Cooper), not a data quality problem.
- The end-to-end postcode -> MP pipeline works correctly on real postcodes we can verify
  manually: `2088` (Mosman) and `2060` (North Sydney area) both correctly resolve to
  Warringah -> **Zali Steggall**; `3000` (Melbourne CBD) -> **Adam Bandt**;
  `6000` (Perth CBD) -> **Patrick Gorman**; `2600` (Canberra) -> **Alicia Payne**. All
  match the real-world current MPs for those seats.
- **Caveat for the full app:** this lookup needs periodic refreshing as boundaries
  redraw — it reflects *current* electorates, so very old Hansard electorates that no
  longer exist won't have a live postcode mapping.

## 3. Build a working sample for topic modelling

**Why sample?** 362K full speeches is too slow to iterate on for topic modelling /
embedding on a laptop CPU. We take a *stratified* sample — up to 700 speeches per
year — so every year from 1998-2025 is represented equally, avoiding bias toward
recent, over-represented years. We also restrict to "substantive" speeches
(80-3000 words) so one-line interjection-style remarks and outlier mega-speeches
don't dominate.

In [ ]:
substantive = clean[(clean["word_count"] >= 80) & (clean["word_count"] <= 3000)].copy()
print(f"Substantive speeches (80-3000 words): {len(substantive):,}")

sample = (
    substantive.groupby("year", group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), 700), random_state=42))
    .reset_index(drop=True)
)
print(f"Stratified sample size: {len(sample):,}")
sample.head()

### Insights — Section 3

- 237,327 of the 362,203 clean speeches (**65%**) are "substantive" (80-3000 words) —
  the rest are short procedural remarks, single-sentence interjections-turned-speeches,
  or similar.
- The stratified sample (~19,600 speeches, ~700/year) keeps every year equally
  represented, which matters because Parliament's sitting patterns and Hansard
  recording conventions have changed over 27 years — an unweighted random sample would
  have over-represented recent years.

## 4. Topic modelling with LDA (unsupervised)

**Technique: Latent Dirichlet Allocation (LDA)** — taught in Session 2. This discovers
topics *without* any labelled data: it assumes each speech is a mixture of latent
topics, and each topic is a distribution over words, then infers both from the corpus.

We remove standard English stopwords **plus** parliamentary procedural filler words
("member", "House", "Speaker", "motion"...) which would otherwise dominate every topic
and hide the actual substance.

In [ ]:
stop_words = set(stopwords.words("english"))

# parliamentary procedural filler words that pollute topics if left in
PARL_STOP = {
    "member", "members", "house", "government", "minister", "speaker",
    "mr", "mrs", "ms", "dr", "honourable", "hon", "today", "would",
    "also", "said", "one", "us", "get", "going", "want", "know", "think",
    "like", "time", "people", "australia", "australian", "australians",
    "parliament", "bill", "motion", "committee", "deputy", "president",
    "senate", "point", "question", "answer", "matter", "issue", "way",
    "make", "made", "take", "taken", "many", "much", "well", "actually",
}
stop_words |= PARL_STOP

TOKEN_RE = re.compile(r"[a-zA-Z]{3,}")

def tokenize(text):
    tokens = TOKEN_RE.findall(text.lower())
    return [t for t in tokens if t not in stop_words]

print("Tokenizing...")
sample["tokens"] = sample["body"].apply(tokenize)

dictionary = corpora.Dictionary(sample["tokens"])
dictionary.filter_extremes(no_below=10, no_above=0.5)  # drop very rare and very common words
corpus = [dictionary.doc2bow(t) for t in sample["tokens"]]
print(f"Vocabulary size after filtering: {len(dictionary)}")

In [ ]:
N_TOPICS = 10

print(f"Training LDA with {N_TOPICS} topics (this can take a few minutes)...")
lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=N_TOPICS,
               passes=8, random_state=42, chunksize=2000)

print("\n=== Top words per topic ===")
for i, topic in lda.print_topics(num_topics=N_TOPICS, num_words=10):
    words = [w.split("*")[1].strip().strip('"') for w in topic.split(" + ")]
    print(f"Topic {i}: {', '.join(words)}")

coherence_model = CoherenceModel(model=lda, texts=sample["tokens"], dictionary=dictionary,
                                  coherence="c_v", processes=1)
coherence = coherence_model.get_coherence()
print(f"\nCoherence (c_v): {coherence:.4f}")

### Insights — Section 4

- LDA reaches a **coherence score of 0.459** (c_v) — a respectable, human-interpretable
  result for parliamentary text (below ~0.3 is generally considered poor, above ~0.5 is
  strong).
- Most discovered topics map cleanly onto real policy areas: e.g. **Education**
  (schools, students, funding, university), **Economy & Tax** (budget, businesses,
  economy), **Defence** (veterans, department, service), **Regional Infrastructure**
  (roads, electorate, local, million).
- **Notable finding:** 2 of the 10 topics aren't about policy issues at all — they're
  pure **partisan rhetoric** ("Labor, party, opposition, prime minister, say, put,
  back"). This is a genuinely useful, honest finding: raw Hansard contains a
  meaningful amount of political point-scoring, not just policy substance — worth
  stating explicitly on the poster rather than hiding it.

## 5. Zero-shot issue classification (embeddings)

**Technique: zero-shot classification via sentence embeddings** (Sessions 7-8 concepts:
zero-shot prompting + embeddings/semantic search). Unlike LDA, this tags each speech
against a *fixed, human-chosen* list of issues (Health, Economy, Immigration...) — more
useful for a "topics my MP talks about" feature than open-ended LDA topics, but it
needs no labelled training data either.

**Method:** embed a short description of each issue category, embed each speech, and
assign each speech to whichever category is most similar (cosine similarity).

In [ ]:
ISSUE_TAXONOMY = {
    "Health": "healthcare, hospitals, Medicare, mental health, aged care, medical research, doctors and nurses",
    "Economy & Tax": "the economy, taxation, budget, inflation, interest rates, jobs and employment, wages, cost of living",
    "Immigration & Border Protection": "immigration, asylum seekers, refugees, visas, border security, citizenship",
    "Environment & Climate": "climate change, emissions, renewable energy, environment protection, national parks, biodiversity",
    "Education": "schools, universities, students, teachers, early childhood education, vocational training",
    "Defence & National Security": "defence force, military, national security, terrorism, veterans, foreign conflict",
    "Housing": "housing affordability, homelessness, rental market, first home buyers, property prices",
    "Social Services & Welfare": "social security, welfare payments, disability support, pensions, centrelink, family payments",
    "Infrastructure & Transport": "roads, rail, public transport, infrastructure investment, airports, regional connectivity",
    "Indigenous Affairs": "Indigenous Australians, Aboriginal and Torres Strait Islander communities, native title, reconciliation, the Voice",
    "Agriculture & Regional": "agriculture, farmers, drought, regional communities, rural industries, water management",
    "Justice & Law": "criminal justice, courts, police, law reform, human rights, legal system",
}

print("Loading sentence-transformer model (all-MiniLM-L6-v2)...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

issue_names = list(ISSUE_TAXONOMY.keys())
issue_embeddings = embed_model.encode(list(ISSUE_TAXONOMY.values()), normalize_embeddings=True)

print("Embedding speeches (first 800 chars each, for speed)...")
texts = sample["body"].str.slice(0, 800).tolist()
speech_embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

raw_sims = speech_embeddings @ issue_embeddings.T  # cosine similarity (vectors are normalized)
print("Done.")

**Problem we found and fixed:** taking the raw argmax of cosine similarity badly
over-assigns some categories — e.g. "Indigenous Affairs" alone captured ~30% of all
speeches, including completely unrelated ones (a copyright-law speech got tagged
Indigenous Affairs). This isn't random noise: raw cosine similarity to a category
description isn't comparable *across* categories, since some category embeddings sit
in "denser" regions of the embedding space and attract false positives even at low
absolute similarity.

**Fix:** (1) z-score calibrate each category's similarity distribution across the
corpus before taking argmax, and (2) only trust a label if it clears both a calibrated
*and* an absolute similarity threshold — otherwise mark the speech "Other / Uncertain"
rather than force-fitting it into one of only 12 categories (many speeches are about
topics outside this list entirely — copyright, foreign affairs, industrial relations,
arts, sport...).

In [ ]:
# calibrate: z-score each category's similarity distribution across the corpus
mu = raw_sims.mean(axis=0, keepdims=True)
sigma = raw_sims.std(axis=0, keepdims=True)
calibrated = (raw_sims - mu) / sigma

best_idx = calibrated.argmax(axis=1)
best_z = calibrated.max(axis=1)
best_raw = raw_sims[np.arange(len(raw_sims)), best_idx]

sample["issue_raw"] = [issue_names[i] for i in best_idx]
sample["issue_zscore"] = best_z
sample["issue_raw_sim"] = best_raw

# only trust the label when BOTH the calibrated z-score and raw cosine similarity
# clear a threshold, else mark as Other/Uncertain
Z_THRESHOLD = 1.0
RAW_THRESHOLD = 0.28
sample["issue"] = np.where(
    (sample["issue_zscore"] >= Z_THRESHOLD) & (sample["issue_raw_sim"] >= RAW_THRESHOLD),
    sample["issue_raw"], "Other / Uncertain"
)

print(f"Confident assignments: {(sample['issue'] != 'Other / Uncertain').sum():,} "
      f"({(sample['issue'] != 'Other / Uncertain').mean():.1%})")
print("\n=== Issue distribution (calibrated + thresholded) ===")
print(sample["issue"].value_counts())

### Insights — Section 5

- Naive (uncalibrated) zero-shot assignment is badly biased: one category alone
  absorbed **~30%** of all speeches regardless of actual topic.
- After z-score calibration + a confidence threshold, only **41.8%** of substantive
  speeches are confidently classified into one of our 12 issues — the rest are
  correctly flagged **"Other/Uncertain"** rather than force-fit. Manually checking a
  sample of the "uncertain" bucket confirms this is the right call: it correctly
  filtered out a UN Security Council speech, an audit-report review, and a valedictory
  speech — none of which fit our fixed issue list.
- This finding is itself an insight worth stating on the poster: **most of what MPs
  say doesn't fit a short list of "big" issues** — it's the unglamorous, specific,
  local stuff that never makes headlines. That's the whole premise behind Parlianote.

## 6. Evaluating classification accuracy

**Technique: manual/human evaluation** (Session 7's "hybrid: code + model + human"
best practice). There's no labelled ground truth for this dataset, so we manually grade
a random sample of confident classifications for correctness. This cell prints the
sample for grading — grade it yourself (or as a team) by reading each speech and
marking correct/incorrect, matching how we validated it during development.

In [ ]:
confident = sample[sample["issue"] != "Other / Uncertain"]
eval_sample = confident.sample(40, random_state=99)[
    ["issue", "issue_zscore", "issue_raw_sim", "displayName", "electorate", "year", "body"]
]

for i, (_, row) in enumerate(eval_sample.iterrows()):
    print(f"\n#{i+1} [{row['issue']}] {row['displayName']} ({row['electorate']}, {row['year']})")
    print(f"  {row['body'][:300]}")

# Save for the team to grade collaboratively (add a 'correct' column: 1/0)
eval_sample.to_csv("../data/eval_sample_40.csv", index=False)
print("\nSaved to data/eval_sample_40.csv for team grading.")

### Insights — Section 6

- Manually grading a random sample of 40 confident classifications gave
  **31/40 correct = 77.5% precision**.
- **5 of the 9 errors** specifically involved "Indigenous Affairs" being wrongly
  applied to unrelated content (a sports-club speech, a jobs summit, community radio,
  a bushfire response) — a *specific, nameable* weakness, not random noise. This is a
  stronger thing to report than a vague accuracy number: we diagnosed *why* the model
  fails, not just *that* it fails sometimes.
- **Recommendation for the full build:** tighten the "Indigenous Affairs" category
  description to be more specific (e.g. explicitly reference the Voice referendum,
  native title claims, Closing the Gap) to reduce this bias.

## 7. Grounded summarisation with RAG

**Technique: Retrieval-Augmented Generation (RAG)** — taught in Session 8. This is the
technical heart of Parlianote's "linked to the official record" promise: instead of
letting a model generate a summary from memory (which can hallucinate), we retrieve the
*actual* relevant passages first, then only summarise from those, with every claim
traceable back to a dated, real excerpt.

**Pipeline (as taught):** chunk documents (500 chars, 100-char overlap) -> embed each
chunk -> index in FAISS -> retrieve top-k chunks for a query -> generate an answer
grounded only in what was retrieved.

We build this for one MP as a worked example — Zali Steggall (Warringah) — since she
has a large volume of speeches and is well-known for hers on integrity/climate.

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

MP_NAME = "Steggall, Zali"
mp_speeches = clean[clean["displayName"] == MP_NAME].copy()
print(f"Speeches by {MP_NAME}: {len(mp_speeches)}")

records = []
for _, row in mp_speeches.iterrows():
    for i, chunk in enumerate(chunk_text(row["body"])):
        if len(chunk.strip()) < 50:
            continue
        records.append({
            "chunk_id": f"{row['uniqueID']}_{row['date'].date()}_{i}",
            "electorate": row["electorate"],
            "displayName": row["displayName"],
            "date": row["date"],
            "chunk_text": chunk,
        })

chunks_df = pd.DataFrame(records)
print(f"Total chunks: {len(chunks_df):,}")

print("Embedding chunks...")
chunk_embeddings = embed_model.encode(chunks_df["chunk_text"].tolist(), batch_size=64,
                                       show_progress_bar=True, normalize_embeddings=True)
chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")

rag_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])  # inner product == cosine sim (vectors normalized)
rag_index.add(chunk_embeddings)
print(f"FAISS index size: {rag_index.ntotal}")

In [ ]:
def retrieve(query, k=5):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = rag_index.search(q_emb, k)
    results = chunks_df.iloc[idxs[0]].copy()
    results["score"] = scores[0]
    return results

# retrieval demo across a range of topics
test_queries = [
    "climate change and emissions targets",
    "political donations and integrity",
    "gambling advertising reform",
    "mental health funding",
]
for q in test_queries:
    print(f"\n=== Query: '{q}' ===")
    for _, r in retrieve(q, k=3).iterrows():
        print(f"  [score={r['score']:.3f}, {r['date'].date()}] {r['chunk_text'][:180].strip()}...")

### Insights — Section 7

- Retrieval quality is strong: in a spot check across **8 diverse test queries**
  (climate, integrity, banking, renewable energy, small business, EVs, gambling
  reform, Ukraine), **24/24 (100%) of top-3 results were genuinely on-topic**, with
  healthy similarity scores (0.44-0.74).
- This directly demonstrates the poster's headline feature working on real data: type
  a topic, get real quotes from what the MP actually said, each with a date attached —
  not a generic AI paraphrase.
- **A grounded summary example** (from querying "gambling advertising reform" and
  writing a summary using only the retrieved passages): *"Zali Steggall has repeatedly
  criticised the government's slow progress on gambling advertising reform. A
  parliamentary inquiry ('You win some, you lose more,' June 2023) unanimously
  recommended restricting gambling advertising; the government accepted the
  recommendations but took years to act. She's argued this is a distraction — the
  government moved quickly on a separate age-verification bill while stalling on the
  ad ban, despite polling showing 70%+ public support for one."* Every claim in this
  summary was manually checked against the retrieved source chunks and is faithfully
  supported — proving the "grounded, not hallucinated" concept works.
- **Bonus finding:** the retrieved passages show Steggall's language on this issue
  *escalating* between Oct 2023 ("slow timeline") and Nov 2024 ("refused to take
  action") — exactly the "how their focus has changed over time" feature from the
  original pitch, demonstrated on real dated data.

## 8. Summary of results

| Component | Technique | Result |
|---|---|---|
| Data cleaning | Filtering / preprocessing | 362,203 clean speeches from 647,852 raw rows |
| Postcode -> MP | Data join | 86% electorate coverage; verified correct on 6 real postcodes |
| Topic discovery | LDA (unsupervised) | Coherence 0.459; mostly clean, human-nameable topics |
| Issue classification | Zero-shot embeddings + calibration | 77.5% precision @ 41.8% coverage; named limitation (Indigenous Affairs bias) |
| Grounded summarisation | RAG (FAISS + embeddings) | 100% relevant retrieval (24/24 test queries); faithful, citable summaries |

**Known limitations (worth stating honestly on the poster):**
1. The issue classifier only confidently tags ~42% of speeches — most parliamentary
   speech doesn't fit a short fixed issue list (which is itself a meaningful finding).
2. "Indigenous Affairs" has a specific, diagnosed bias toward false positives on
   generic community-praise language.
3. The postcode -> electorate mapping reflects *current* boundaries and will need
   periodic updates as electorates are redrawn.

**Next steps for the full build:** extend the RAG pipeline to all MPs (not just one),
wrap it in a simple web app (Streamlit, per Session 8's template) with the postcode
search box, and consider swapping the fixed issue taxonomy for LDA-discovered topics
where the fixed list runs out of coverage.